In [1]:
import numpy as np
np.__version__ 

'2.1.3'

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import autokeras as ak
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from sklearn.model_selection import train_test_split # Import train_test_split

# --- Parâmetros de Configuração ---
IMG_HEIGHT = 33
IMG_WIDTH = 81
NUM_CLASSES = 3
BATCH_SIZE = 16
MAX_TRIALS = 5
EPOCHS_PER_TRIAL = 10

label_map = {'rest': 0, 'left': 1, 'right': 2}
class_names = ['rest', 'left', 'right']

# --- Função para carregar e pré-processar um único espectrograma (.npy) ---
def load_spectrogram_data(filepath_c3, filepath_c4, label):
    spectrogram_c3 = np.load(filepath_c3.numpy()).astype(np.float32)
    spectrogram_c4 = np.load(filepath_c4.numpy()).astype(np.float32)

    min_c3, max_c3 = np.min(spectrogram_c3), np.max(spectrogram_c3)
    spectrogram_c3 = (spectrogram_c3 - min_c3) / (max_c3 - min_c3 + 1e-8) if (max_c3 - min_c3) > 1e-8 else spectrogram_c3

    min_c4, max_c4 = np.min(spectrogram_c4), np.max(spectrogram_c4)
    spectrogram_c4 = (spectrogram_c4 - min_c4) / (max_c4 - min_c4 + 1e-8) if (max_c4 - min_c4) > 1e-8 else spectrogram_c4

    combined_spectrogram = np.stack([spectrogram_c3, spectrogram_c4], axis=-1)
    
    return combined_spectrogram, label

# --- Função para Criar o Dataset (Adaptada para AutoKeras com train_test_split) ---
def create_dataset_from_directories_npy(data_root_dir, is_training_data=True, validation_split_ratio=0.2):
    all_filepaths_c3 = []
    all_filepaths_c4 = []
    all_labels = []

    print(f"Tentando ler do diretório raiz: {data_root_dir}")
    if not os.path.exists(data_root_dir):
        print(f"Erro: O diretório raiz dos dados não existe: {data_root_dir}")
        return None, None
            
    for class_name in label_map.keys():
        class_dir = os.path.join(data_root_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"  Aviso: O diretório da classe '{class_name}' não existe em {data_root_dir}. Pulando.")
            continue 

        for img_filename in os.listdir(class_dir):
            if img_filename.endswith('_C3.npy'):
                filepath_c3 = os.path.join(class_dir, img_filename)
                filepath_c4 = filepath_c3.replace('_C3.npy', '_C4.npy')
                
                if os.path.exists(filepath_c4):
                    all_filepaths_c3.append(filepath_c3)
                    all_filepaths_c4.append(filepath_c4)
                    all_labels.append(label_map[class_name])
    
    print(f"Total de arquivos C3 encontrados: {len(all_filepaths_c3)}")
    print(f"Total de arquivos C4 encontrados: {len(all_filepaths_c4)}")
    print(f"Total de rótulos encontrados: {len(all_labels)}")

    if len(all_filepaths_c3) == 0:
        print("Erro: Nenhum par de espectrogramas C3/C4 encontrado. O dataset está vazio.")
        return None, None

    if is_training_data:
        # Use train_test_split para dividir os caminhos dos arquivos e rótulos
        train_filepaths_c3, val_filepaths_c3, \
        train_filepaths_c4, val_filepaths_c4, \
        train_labels, val_labels = train_test_split(
            all_filepaths_c3, all_filepaths_c4, all_labels, 
            test_size=validation_split_ratio, 
            random_state=42, # Para reprodutibilidade
            stratify=all_labels # Para manter a proporção de classes
        )

        train_ds = tf.data.Dataset.from_tensor_slices(((tf.constant(train_filepaths_c3, dtype=tf.string), 
                                                        tf.constant(train_filepaths_c4, dtype=tf.string)), 
                                                       tf.constant(train_labels, dtype=tf.int32)))
        train_ds = train_ds.map(lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
                                num_parallel_calls=tf.data.AUTOTUNE)
        train_ds = train_ds.map(lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
                                num_parallel_calls=tf.data.AUTOTUNE)
        train_ds = train_ds.shuffle(buffer_size=len(train_filepaths_c3)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        val_ds_split = tf.data.Dataset.from_tensor_slices(((tf.constant(val_filepaths_c3, dtype=tf.string), 
                                                            tf.constant(val_filepaths_c4, dtype=tf.string)), 
                                                           tf.constant(val_labels, dtype=tf.int32)))
        val_ds_split = val_ds_split.map(lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
                                         num_parallel_calls=tf.data.AUTOTUNE)
        val_ds_split = val_ds_split.map(lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
                                       num_parallel_calls=tf.data.AUTOTUNE)
        val_ds_split = val_ds_split.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        return train_ds, val_ds_split
    else:
        # Para o dataset de validação externo (teste), não há split
        filepaths_c3_tensor = tf.constant(all_filepaths_c3, dtype=tf.string)
        filepaths_c4_tensor = tf.constant(all_filepaths_c4, dtype=tf.string)
        labels_tensor = tf.constant(all_labels, dtype=tf.int32)

        dataset = tf.data.Dataset.from_tensor_slices(((filepaths_c3_tensor, filepaths_c4_tensor), labels_tensor))
        dataset = dataset.map(lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
                              num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.map(lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
                              num_parallel_calls=tf.data.AUTOTUNE)
        return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- Aplicação do AutoKeras ---
if __name__ == '__main__':
    selected_window_ms = 200

    train_data_root_dir = f'unified_spectrograms_{selected_window_ms}ms'
    val_data_root_dir = f'validation_unified_spectrograms_{selected_window_ms}ms'

    print(f"\n--- Aplicando AutoKeras para a janela de {selected_window_ms}ms ---")

    train_ds, ak_val_ds = create_dataset_from_directories_npy(train_data_root_dir, is_training_data=True, validation_split_ratio=0.2)
    final_val_ds = create_dataset_from_directories_npy(val_data_root_dir, is_training_data=False)

    if train_ds is None or tf.data.experimental.cardinality(train_ds).numpy() == 0:
        print(f"Não há dados de treinamento suficientes em {train_data_root_dir}. AutoKeras abortado.")
    elif final_val_ds is None or tf.data.experimental.cardinality(final_val_ds).numpy() == 0:
        print(f"Não há dados de validação suficientes em {val_data_root_dir}. AutoKeras abortado.")
    else:
        print(f"Dados de treinamento carregados de: {train_data_root_dir}")
        print(f"Dados de validação (interna AutoKeras) carregados de: {train_data_root_dir} (split)")
        print(f"Dados de validação (externa final) carregados de: {val_data_root_dir}")

        clf = ak.ImageClassifier(
            overwrite=True,
            max_trials=MAX_TRIALS,
            objective="val_accuracy",
            directory="autokeras_models",
            project_name=f"spectrogram_classifier_{selected_window_ms}ms"
        )

        print("\n--- Iniciando a busca da melhor arquitetura com AutoKeras ---")
        clf.fit(
            train_ds,
            validation_data=ak_val_ds,
            epochs=EPOCHS_PER_TRIAL
        )

        best_model = clf.export_model()
        
        print("\n--- Melhor Modelo Encontrado pelo AutoKeras ---")
        best_model.summary()

        print("\n--- Avaliação Final do Melhor Modelo no Dataset de Validação Externo ---")
        loss, accuracy = best_model.evaluate(final_val_ds, verbose=0)
        print(f"\nResultados da Avaliação Final ({selected_window_ms}ms):")
        print(f"  Perda no conjunto de validação externo: {loss:.4f}")
        print(f"  Acurácia no conjunto de validação externo: {accuracy*100:.2f}%")

        best_model_save_path = f'autokeras_best_model_{selected_window_ms}ms.keras'
        best_model.save(best_model_save_path)
        print(f"Melhor modelo AutoKeras salvo como '{best_model_save_path}'")

        print("\n--- Coletando previsões para análise detalhada no dataset de validação externo ---")
        all_predictions = []
        all_true_labels = []

        for spectrograms_batch, labels_batch in final_val_ds:
            predictions_batch = best_model.predict(spectrograms_batch, verbose=0)
            predicted_classes_batch = np.argmax(predictions_batch, axis=1)
            
            all_predictions.extend(predicted_classes_batch)
            all_true_labels.extend(labels_batch.numpy())
        
        all_predictions = np.array(all_predictions)
        all_true_labels = np.array(all_true_labels)

        print("\n--- Gerando Matriz de Confusão ---")
        cm = confusion_matrix(all_true_labels, all_predictions)
        print(cm)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Matriz de Confusão do AutoKeras ({selected_window_ms}ms)')
        plt.xlabel('Rótulo Predito')
        plt.ylabel('Rótulo Verdadeiro')
        plt.show()

        print("\n--- Relatório de Classificação ---")
        report = classification_report(all_true_labels, all_predictions, target_names=class_names, zero_division=0)
        print(report)

        print("\n--- Visualizando alguns espectrogramas com previsões (C3 e C4) do melhor modelo AutoKeras ---")
        num_samples_to_plot = 5
        
        final_val_ds_for_plot = create_dataset_from_directories_npy(val_data_root_dir, is_training_data=False)
        
        fig, axes = plt.subplots(num_samples_to_plot, 2, figsize=(14, 4 * num_samples_to_plot), sharex=True)
        axes = axes.flatten()

        sample_count = 0
        for spectrograms_batch, labels_batch in final_val_ds_for_plot.take(num_samples_to_plot // BATCH_SIZE + 1):
            if sample_count >= num_samples_to_plot:
                break
            
            predictions_batch = best_model.predict(spectrograms_batch, verbose=0)

            for i in range(min(BATCH_SIZE, num_samples_to_plot - sample_count)):
                spec_c3 = spectrograms_batch[i,:,:,0].numpy().T
                spec_c4 = spectrograms_batch[i,:,:,1].numpy().T
                
                true_label_idx = labels_batch[i].numpy()
                predicted_label_idx = np.argmax(predictions_batch[i])

                ax_c3 = axes[sample_count * 2]
                ax_c3.imshow(spec_c3, aspect='auto', origin='lower', cmap='viridis')
                ax_c3.set_title(f'C3 - Real: {class_names[true_label_idx]} Pred: {class_names[predicted_label_idx]}')
                ax_c3.set_ylabel('Frequência [Hz]')
                if sample_count == num_samples_to_plot - 1:
                    ax_c3.set_xlabel('Tempo [s]')
                ax_c3.set_ylim([8, 30])

                ax_c4 = axes[sample_count * 2 + 1]
                ax_c4.imshow(spec_c4, aspect='auto', origin='lower', cmap='viridis')
                ax_c4.set_title(f'C4 - Real: {class_names[true_label_idx]} Pred: {class_names[predicted_label_idx]}')
                if sample_count == num_samples_to_plot - 1:
                    ax_c4.set_xlabel('Tempo [s]')
                ax_c4.set_ylim([8, 30])

                sample_count += 1
                if sample_count >= num_samples_to_plot:
                    break
            
        plt.tight_layout()
        plt.show()

Trial 1 Complete [00h 06m 52s]
val_accuracy: 0.5322222113609314

Best val_accuracy So Far: 0.5322222113609314
Total elapsed time: 00h 06m 52s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
resnet            |vanilla           |image_block_1/block_type
True              |True              |image_block_1/normalize
True              |False             |image_block_1/augment
True              |None              |image_block_1/image_augmentation_1/horizontal_flip
True              |None              |image_block_1/image_augmentation_1/vertical_flip
0                 |None              |image_block_1/image_augmentation_1/contrast_factor
0                 |None              |image_block_1/image_augmentation_1/rotation_factor
0.1               |None              |image_block_1/image_augmentation_1/translation_factor
0                 |None              |image_block_1/image_augmentation_1/zoom_factor
False             |None              |image_block_1/res_net_bl

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import autokeras as ak
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import seaborn as sns

# --- Parâmetros de Configuração ---
IMG_HEIGHT = 33
IMG_WIDTH = 81
NUM_CLASSES = 3
BATCH_SIZE = 32
MAX_TRIALS = 10 
EPOCHS_PER_TRIAL = 20
SEED = 42 # Para reprodutibilidade

# Mapeamento e Nomes de Classes
label_map = {'rest': 0, 'left': 1, 'right': 2}
class_names = ['rest', 'left', 'right']

# --- Funções de Processamento de Dados ---

def load_spectrogram_data(filepath_c3, filepath_c4, label):
    """Carrega e pré-processa um par de espectrogramas."""
    spectrogram_c3 = np.load(filepath_c3.numpy()).astype(np.float32)
    spectrogram_c4 = np.load(filepath_c4.numpy()).astype(np.float32)

    min_c3, max_c3 = np.min(spectrogram_c3), np.max(spectrogram_c3)
    if (max_c3 - min_c3) > 1e-8:
        spectrogram_c3 = (spectrogram_c3 - min_c3) / (max_c3 - min_c3 + 1e-8)

    min_c4, max_c4 = np.min(spectrogram_c4), np.max(spectrogram_c4)
    if (max_c4 - min_c4) > 1e-8:
        spectrogram_c4 = (spectrogram_c4 - min_c4) / (max_c4 - min_c4 + 1e-8)

    combined_spectrogram = np.stack([spectrogram_c3, spectrogram_c4], axis=-1)
    
    return combined_spectrogram, label

def build_tf_dataset(files_c3, files_c4, labels):
    """Constrói um tf.data.Dataset a partir de listas de caminhos e rótulos."""
    dataset = tf.data.Dataset.from_tensor_slices(((files_c3, files_c4), labels))
    dataset = dataset.map(
        lambda x, y: tf.py_function(load_spectrogram_data, [x[0], x[1], y], (tf.float32, tf.int32)),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.map(
        lambda spec, label: (tf.ensure_shape(spec, (IMG_HEIGHT, IMG_WIDTH, 2)), tf.ensure_shape(label, ())),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return dataset

# --- Lógica Principal com AutoKeras ---
if __name__ == '__main__':
    selected_window_ms = 200
    # Unimos todos os dados em um único diretório para uma divisão mais robusta
    all_data_root_dir = f'unified_spectrograms_{selected_window_ms}ms'

    print(f"\n--- 1. Carregando todos os caminhos de arquivos de '{all_data_root_dir}' ---")
    all_filepaths_c3, all_filepaths_c4, all_labels = [], [], []
    if os.path.exists(all_data_root_dir):
        for class_name, label_idx in label_map.items():
            class_dir = os.path.join(all_data_root_dir, class_name)
            if not os.path.exists(class_dir): continue
            for fname in os.listdir(class_dir):
                if fname.endswith('_C3.npy'):
                    fp_c3 = os.path.join(class_dir, fname)
                    fp_c4 = fp_c3.replace('_C3.npy', '_C4.npy')
                    if os.path.exists(fp_c4):
                        all_filepaths_c3.append(fp_c3)
                        all_filepaths_c4.append(fp_c4)
                        all_labels.append(label_idx)
    
    if not all_filepaths_c3:
        print("Nenhum dado encontrado. Abortando.")
    else:
        print(f"Total de {len(all_labels)} amostras encontradas.")

        # --- 2. Divisão Robusta em Três Conjuntos: Treino, Validação e Teste ---
        print("\n--- 2. Dividindo dados em conjuntos de Treino, Validação e Teste ---")
        
        # Primeiro, separamos o conjunto de teste (15% do total)
        train_val_c3, test_c3, train_val_c4, test_c4, train_val_labels, test_labels = train_test_split(
            all_filepaths_c3, all_filepaths_c4, all_labels,
            test_size=0.15, random_state=SEED, stratify=all_labels
        )

        # Depois, dividimos o restante em treino (70%) e validação interna (15%)
        # A nova test_size é 0.15 / (1 - 0.15) = 0.1765 para obter a proporção correta
        train_c3, val_c3, train_c4, val_c4, train_labels, val_labels = train_test_split(
            train_val_c3, train_val_c4, train_val_labels,
            test_size=0.1765, random_state=SEED, stratify=train_val_labels
        )
        
        print(f"Divisão final: {len(train_labels)} treino, {len(val_labels)} validação, {len(test_labels)} teste.")

        # --- 3. Construção dos tf.data.Datasets ---
        print("\n--- 3. Construindo os objetos tf.data.Dataset ---")
        train_ds = build_tf_dataset(train_c3, train_c4, train_labels)
        val_ds = build_tf_dataset(val_c3, val_c4, val_labels)
        test_ds = build_tf_dataset(test_c3, test_c4, test_labels)

        # Otimização dos pipelines
        train_ds = train_ds.shuffle(len(train_labels)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        # --- 4. Configuração e Execução do AutoKeras ---
        print("\n--- 4. Configurando e iniciando a busca com AutoKeras ---")
        clf = ak.ImageClassifier(
            overwrite=True,
            max_trials=MAX_TRIALS,
            objective="val_accuracy",
            directory="autokeras_models_v2",
            project_name=f"spectrogram_classifier_{selected_window_ms}ms"
        )

        # A busca do AutoKeras usa o `val_ds` para encontrar o melhor modelo
        clf.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_PER_TRIAL
        )

        # --- 5. Exportação e Avaliação Final ---
        print("\n--- 5. Exportando e avaliando o melhor modelo no conjunto de TESTE ---")
        best_model = clf.export_model()
        best_model.summary()
        
        loss, accuracy = best_model.evaluate(test_ds)
        print(f"\nAcurácia final no conjunto de TESTE (dados não vistos): {accuracy*100:.2f}%")
        
        best_model.save(f'autokeras_best_model_{selected_window_ms}ms_v2.keras')

        # --- 6. Análise de Resultados Otimizada ---
        print("\n--- 6. Gerando análise de resultados detalhada ---")
        predictions = best_model.predict(test_ds)
        predicted_classes = np.argmax(predictions, axis=1)
        
        # Usamos os rótulos do conjunto de teste que já temos, sem percorrer o dataset novamente
        true_classes = np.array(test_labels) 

        # Matriz de Confusão
        cm = confusion_matrix(true_classes, predicted_classes)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Matriz de Confusão do AutoKeras (Teste Final) ({selected_window_ms}ms)')
        plt.ylabel('Rótulo Verdadeiro')
        plt.xlabel('Rótulo Predito')
        plt.show()

        # Relatório de Classificação
        print("\n--- Relatório de Classificação (Teste Final) ---")
        print(classification_report(true_classes, predicted_classes, target_names=class_names, zero_division=0))

Trial 1 Complete [00h 08m 40s]
val_accuracy: 0.5

Best val_accuracy So Far: 0.5
Total elapsed time: 00h 08m 40s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
resnet            |vanilla           |image_block_1/block_type
True              |True              |image_block_1/normalize
True              |False             |image_block_1/augment
True              |None              |image_block_1/image_augmentation_1/horizontal_flip
True              |None              |image_block_1/image_augmentation_1/vertical_flip
0                 |None              |image_block_1/image_augmentation_1/contrast_factor
0                 |None              |image_block_1/image_augmentation_1/rotation_factor
0.1               |None              |image_block_1/image_augmentation_1/translation_factor
0                 |None              |image_block_1/image_augmentation_1/zoom_factor
False             |None              |image_block_1/res_net_block_1/pretrained
resnet50     